In [1]:
file_path = "/content/hostel_bois.txt"

with open(file_path, "r", encoding="utf-8") as f:
    lines = f.readlines()

print("Total lines:", len(lines))

Total lines: 3178


In [3]:
from datetime import datetime

messages = []
system_messages = 0
media_messages = 0
deleted_messages = 0

for line in lines:
    line = line.strip()

    if not line:
        continue

    # Check if line starts with date
    if len(line) >= 18 and line[2] == "/" and line[5] == "/":

        try:
            date_time = line[:15]
            rest = line[18:]

            dt = datetime.strptime(date_time, "%d/%m/%y, %H:%M")

            # System message
            if ": " not in rest:
                system_messages += 1
                continue

            sender_text = rest.split(": ", 1)

            sender = sender_text[0]
            text = sender_text[1]

            # Media message
            if text == "<Media omitted>":
                media_messages += 1

                messages.append({
                    "datetime": dt,
                    "date": dt.date(),
                    "time": dt.time(),
                    "sender": sender,
                    "text": text,
                    "type": "media"
                })

            # Deleted message
            elif text == "This message was deleted":
                deleted_messages += 1

                messages.append({
                    "datetime": dt,
                    "date": dt.date(),
                    "time": dt.time(),
                    "sender": sender,
                    "text": text,
                    "type": "deleted"
                })

            # Normal message
            else:
                messages.append({
                    "datetime": dt,
                    "date": dt.date(),
                    "time": dt.time(),
                    "sender": sender,
                    "text": text,
                    "type": "normal"
                })

        except ValueError:
            system_messages += 1


print("Parsed messages:", len(messages))
print("System messages:", system_messages)
print("Media messages:", media_messages)
print("Deleted messages:", deleted_messages)



Parsed messages: 3174
System messages: 4
Media messages: 32
Deleted messages: 15


In [4]:
participants = set()

for message in messages:
    participants.add(message["sender"])

print("Participants:")
for person in sorted(participants):
    print(person)

print("\nNumber of participants:", len(participants))

Participants:
Aman
Karan
Neha
Priya
Rahul
Vikas

Number of participants: 6


In [5]:
# Count messages for each participant

message_counts = {}

for person in participants:
    message_counts[person] = 0

for message in messages:
    message_counts[message["sender"]] += 1

# Sort from highest to lowest
sorted_counts = sorted(
    message_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

print("GROUP MESSAGE OVERVIEW")
print("-" * 30)

for person, count in sorted_counts:
    print(f"{person}: {count} messages")

GROUP MESSAGE OVERVIEW
------------------------------
Rahul: 953 messages
Priya: 718 messages
Neha: 635 messages
Aman: 490 messages
Karan: 354 messages
Vikas: 24 messages


In [6]:
# Total messages
total_messages = len(messages)

# Find first and last date
dates = []

for message in messages:
    dates.append(message["date"])

start_date = min(dates)
end_date = max(dates)

# Calculate total days
total_days = (end_date - start_date).days + 1

print("GROUP OVERVIEW")
print("=" * 40)
print(f"Total messages : {total_messages}")
print(f"Start date     : {start_date}")
print(f"End date       : {end_date}")
print(f"Total days     : {total_days}")
print(f"Participants   : {len(participants)}")
print()

print("Messages by participant:")
print("-" * 40)

for person, count in sorted_counts:
    print(f"{person:<10} : {count}")

GROUP OVERVIEW
Total messages : 3174
Start date     : 2024-04-01
End date       : 2024-05-30
Total days     : 60
Participants   : 6

Messages by participant:
----------------------------------------
Rahul      : 953
Priya      : 718
Neha       : 635
Aman       : 490
Karan      : 354
Vikas      : 24


In [7]:
# Count messages for each day
day_counts = {}

for message in messages:
    date = message["date"]

    if date not in day_counts:
        day_counts[date] = 0

    day_counts[date] += 1


# Find the busiest day
busiest_day = max(day_counts, key=day_counts.get)
busiest_day_count = day_counts[busiest_day]


# Count messages for each hour
hour_counts = {}

for message in messages:
    hour = message["datetime"].hour

    if hour not in hour_counts:
        hour_counts[hour] = 0

    hour_counts[hour] += 1


# Find the busiest hour
busiest_hour = max(hour_counts, key=hour_counts.get)
busiest_hour_count = hour_counts[busiest_hour]


print("MOST ACTIVE DAY & HOUR")
print("=" * 40)

print(f"Busiest day  : {busiest_day}")
print(f"Messages     : {busiest_day_count}")

print()

print(f"Busiest hour : {busiest_hour:02d}:00 - {busiest_hour + 1:02d}:00")
print(f"Messages     : {busiest_hour_count}")

MOST ACTIVE DAY & HOUR
Busiest day  : 2024-05-04
Messages     : 76

Busiest hour : 18:00 - 19:00
Messages     : 248


In [8]:
import numpy as np

# Create 6 x 24 matrix
heatmap = np.zeros((len(participants), 24), dtype=int)

# Keep participants in sorted order
participant_list = sorted(participants)

# Count messages by person and hour
for message in messages:
    sender = message["sender"]
    hour = message["datetime"].hour

    # Media and deleted messages are still messages
    row = participant_list.index(sender)

    heatmap[row][hour] += 1

print("ACTIVITY HEATMAP")
print("=" * 60)

# Print header
print(f"{'Person':<10}", end="")

for hour in range(24):
    print(f"{hour:>4}", end="")

print()

# Print rows
for i in range(len(participant_list)):
    print(f"{participant_list[i]:<10}", end="")

    for hour in range(24):
        print(f"{heatmap[i][hour]:>4}", end="")

    print()

ACTIVITY HEATMAP
Person       0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18  19  20  21  22  23
Aman        54  67  66  60  88   0   0   0   0   0   0   0   0   0  14  11  19   7  16   8  13  11   0  56
Karan        0   0   0   0   0   0   0   4  12  16  20  16  37  25  32  27  27  27  25  32  23  14   9   8
Neha         0   0   0   0   0  19   3  13  36  52  52  22  39  36  27  10  37  47  62  50  45  27  28  30
Priya        0   0   0   0   0   0  13  20  47  65  62  61  57  48  44  29  32  40  38  60  43  32  18   9
Rahul        3  15  17  17  22  10  17  17  24  17  25  15  58  48  45  53  73  49 105  76  41  92  60  54
Vikas        0   0   0   0   0   0   0   1   3   1   1   0   2   2   0   1   1   3   2   2   1   1   1   2


In [9]:
# Find maximum value
max_value = heatmap.max()

# Characters for different activity levels
shades = [" ", "░", "▒", "▓", "█"]

print("GROUPDNA ACTIVITY HEATMAP")
print("=" * 90)

# Hour headings
print(f"{'Person':<10}", end="")

for hour in range(24):
    print(f"{hour:>3}", end="")

print()

# Heatmap
for i, person in enumerate(participant_list):
    print(f"{person:<10}", end="")

    for hour in range(24):
        value = heatmap[i][hour]

        if value == 0:
            level = 0
        else:
            level = int((value / max_value) * 4) + 1
            if level > 4:
                level = 4

        print(f" {shades[level]}", end=" ")

    print()

print()
print("Legend:")
print("  = No activity")
print("░ = Low")
print("▒ = Medium")
print("▓ = High")
print("█ = Very High")

GROUPDNA ACTIVITY HEATMAP
Person      0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
Aman       ▓  ▓  ▓  ▓  █                             ░  ░  ░  ░  ░  ░  ░  ░     ▓ 
Karan                           ░  ░  ░  ░  ░  ▒  ░  ▒  ▒  ▒  ▒  ░  ▒  ░  ░  ░  ░ 
Neha                      ░  ░  ░  ▒  ▒  ▒  ░  ▒  ▒  ▒  ░  ▒  ▒  ▓  ▒  ▒  ▒  ▒  ▒ 
Priya                        ░  ░  ▒  ▓  ▓  ▓  ▓  ▒  ▒  ▒  ▒  ▒  ▒  ▓  ▒  ▒  ░  ░ 
Rahul      ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ░  ▓  ▒  ▒  ▓  ▓  ▒  █  ▓  ▒  █  ▓  ▓ 
Vikas                           ░  ░  ░  ░     ░  ░     ░  ░  ░  ░  ░  ░  ░  ░  ░ 

Legend:
  = No activity
░ = Low
▒ = Medium
▓ = High
█ = Very High


In [11]:
# Common stop words
stop_words = {
    "the", "is", "a", "an", "and", "or", "to", "of",
    "in", "on", "for", "with", "this", "that", "it",
    "are", "was", "be", "i", "you", "we", "he", "she",
    "they", "my", "your", "me", "our", "but", "so",
    "if", "at", "from", "have", "has", "had", "not",
    "do", "did", "will", "can", "just", "all", "as"
}

word_counts = {}

for message in messages:

    # Only normal messages
    if message["type"] != "normal":
        continue

    text = message["text"].lower()

    # Remove punctuation
    punctuation = ".,!?;:'\"()[]{}<>-/\\|@#$%^&*_+=~`"

    for char in punctuation:
        text = text.replace(char, " ")

    words = text.split()

    for word in words:

        # Skip stop words
        if word in stop_words:
            continue

        # Skip single-letter words
        if len(word) <= 1:
            continue

        # Count the word
        if word not in word_counts:
            word_counts[word] = 0

        word_counts[word] += 1


# Sort words by frequency
sorted_words = sorted(
    word_counts.items(),
    key=lambda x: x[1],
    reverse=True
)

print("TOP 10 WORDS")
print("=" * 30)

for word, count in sorted_words[:10]:
    print(f"{word:<15} : {count}")

TOP 10 WORDS
how             : 321
guys            : 318
today           : 292
about           : 274
hai             : 268
am              : 260
his             : 217
everyone        : 203
which           : 202
telling         : 179


In [12]:
from datetime import timedelta

# Sort messages by datetime
sorted_messages = sorted(messages, key=lambda x: x["datetime"])

response_times = {}

for person in participants:
    response_times[person] = []

# Compare each message with the previous message
for i in range(1, len(sorted_messages)):

    previous_message = sorted_messages[i - 1]
    current_message = sorted_messages[i]

    # Only calculate response when different people send messages
    if previous_message["sender"] != current_message["sender"]:

        person = current_message["sender"]

        time_difference = (
            current_message["datetime"] -
            previous_message["datetime"]
        )

        minutes = time_difference.total_seconds() / 60

        response_times[person].append(minutes)


# Calculate average response time
average_response = {}

for person in participants:

    if len(response_times[person]) > 0:
        average_response[person] = (
            sum(response_times[person]) /
            len(response_times[person])
        )
    else:
        average_response[person] = 0


print("AVERAGE RESPONSE TIME")
print("=" * 40)

for person in sorted(
    average_response,
    key=average_response.get
):
    print(
        f"{person:<10} : "
        f"{average_response[person]:.2f} minutes"
    )

AVERAGE RESPONSE TIME
Rahul      : 34.95 minutes
Karan      : 36.62 minutes
Neha       : 39.45 minutes
Priya      : 41.99 minutes
Vikas      : 46.30 minutes
Aman       : 55.36 minutes


In [13]:
# Get all dates in the chat
all_dates = []

for i in range(total_days):
    current_date = start_date + timedelta(days=i)
    all_dates.append(current_date)


silent_streaks = {}

for person in participants:

    # Dates on which the person sent a message
    active_dates = set()

    for message in messages:
        if message["sender"] == person:
            active_dates.add(message["date"])

    longest_streak = 0
    current_streak = 0

    for date in all_dates:

        if date not in active_dates:
            current_streak += 1

            if current_streak > longest_streak:
                longest_streak = current_streak

        else:
            current_streak = 0

    silent_streaks[person] = longest_streak


print("LONGEST SILENT STREAK")
print("=" * 40)

for person in sorted(
    silent_streaks,
    key=silent_streaks.get,
    reverse=True
):
    print(
        f"{person:<10} : "
        f"{silent_streaks[person]} days"
    )

LONGEST SILENT STREAK
Vikas      : 11 days
Rahul      : 0 days
Karan      : 0 days
Priya      : 0 days
Neha       : 0 days
Aman       : 0 days


In [14]:
# ==========================================
# PERSONALITY ARCHETYPE DETECTION
# ==========================================

normal_messages = []

for message in messages:
    if message["type"] == "normal":
        normal_messages.append(message)

# ------------------------------------------
# 1. SPAMMER - Average consecutive burst
# ------------------------------------------

burst_lengths = {}

for person in participants:
    burst_lengths[person] = []

current_sender = None
current_burst = 0

for message in sorted_messages:
    sender = message["sender"]

    if sender == current_sender:
        current_burst += 1
    else:
        if current_sender is not None:
            burst_lengths[current_sender].append(current_burst)

        current_sender = sender
        current_burst = 1

# Add last burst
if current_sender is not None:
    burst_lengths[current_sender].append(current_burst)

average_burst = {}

for person in participants:
    if len(burst_lengths[person]) > 0:
        average_burst[person] = (
            sum(burst_lengths[person])
            / len(burst_lengths[person])
        )
    else:
        average_burst[person] = 0


# ------------------------------------------
# 2. GROUP MOM - Caring keywords
# ------------------------------------------

caring_keywords = [
    "okay",
    "safe",
    "eat",
    "sleep",
    "take care",
    "are you",
    "please",
    "reminder",
    "drink water",
    "don't forget"
]

caring_counts = {}

for person in participants:
    caring_counts[person] = 0

for message in normal_messages:
    text = message["text"].lower()

    for keyword in caring_keywords:
        if keyword in text:
            caring_counts[message["sender"]] += 1


# ------------------------------------------
# 3. NIGHT OWL - 11 PM to 5 AM
# ------------------------------------------

night_percent = {}

for person in participants:

    person_messages = []

    for message in messages:
        if message["sender"] == person:
            person_messages.append(message)

    night_count = 0

    for message in person_messages:
        hour = message["datetime"].hour

        if hour >= 23 or hour < 5:
            night_count += 1

    if len(person_messages) > 0:
        night_percent[person] = (
            night_count / len(person_messages)
        ) * 100
    else:
        night_percent[person] = 0


# ------------------------------------------
# 4. STORYTELLER - Average words/message
# ------------------------------------------

average_words = {}

for person in participants:

    person_messages = []

    for message in normal_messages:
        if message["sender"] == person:
            person_messages.append(message)

    total_words = 0

    for message in person_messages:
        words = message["text"].split()
        total_words += len(words)

    if len(person_messages) > 0:
        average_words[person] = (
            total_words / len(person_messages)
        )
    else:
        average_words[person] = 0


# ------------------------------------------
# 5. DRAMA QUEEN
# ------------------------------------------

uppercase_percent = {}
exclamation_counts = {}

for person in participants:

    eligible_messages = []
    uppercase_messages = 0
    exclamation_messages = 0

    for message in normal_messages:

        if message["sender"] != person:
            continue

        text = message["text"].strip()

        if len(text) >= 3:
            eligible_messages.append(text)

            if text.isupper():
                uppercase_messages += 1

        if "!!" in text:
            exclamation_messages += 1

    if len(eligible_messages) > 0:
        uppercase_percent[person] = (
            uppercase_messages
            / len(eligible_messages)
        ) * 100
    else:
        uppercase_percent[person] = 0

    exclamation_counts[person] = exclamation_messages


# ------------------------------------------
# 6. GHOST - Silent percentage
# ------------------------------------------

silent_percent = {}

for person in participants:

    active_days = set()

    for message in messages:
        if message["sender"] == person:
            active_days.add(message["date"])

    silent_days = total_days - len(active_days)

    silent_percent[person] = (
        silent_days / total_days
    ) * 100


# ------------------------------------------
# 7. COMEDIAN
# ------------------------------------------

funny_words = [
    "lol",
    "lmao",
    "haha",
    "rofl",
    "lmfao"
]

comedian_percent = {}

for person in participants:

    person_messages = []

    for message in normal_messages:
        if message["sender"] == person:
            person_messages.append(message)

    funny_count = 0

    for message in person_messages:
        text = message["text"].lower()

        for word in funny_words:
            if word in text:
                funny_count += 1
                break

    if len(person_messages) > 0:
        comedian_percent[person] = (
            funny_count / len(person_messages)
        ) * 100
    else:
        comedian_percent[person] = 0


# ------------------------------------------
# 8. QUESTION MASTER
# ------------------------------------------

question_percent = {}

for person in participants:

    person_messages = []

    for message in normal_messages:
        if message["sender"] == person:
            person_messages.append(message)

    question_count = 0

    for message in person_messages:
        if message["text"].strip().endswith("?"):
            question_count += 1

    if len(person_messages) > 0:
        question_percent[person] = (
            question_count / len(person_messages)
        ) * 100
    else:
        question_percent[person] = 0


# ------------------------------------------
# DISPLAY ANALYSIS
# ------------------------------------------

print("PERSONALITY ANALYSIS")
print("=" * 60)

for person in sorted(participants):

    print(f"\n{person}")
    print("-" * 40)

    print(
        f"Average burst       : "
        f"{average_burst[person]:.2f}"
    )

    print(
        f"Caring keywords     : "
        f"{caring_counts[person]}"
    )

    print(
        f"Night messages      : "
        f"{night_percent[person]:.2f}%"
    )

    print(
        f"Average words/msg   : "
        f"{average_words[person]:.2f}"
    )

    print(
        f"Uppercase messages  : "
        f"{uppercase_percent[person]:.2f}%"
    )

    print(
        f"Silent days         : "
        f"{silent_percent[person]:.2f}%"
    )

    print(
        f"Funny messages      : "
        f"{comedian_percent[person]:.2f}%"
    )

    print(
        f"Questions           : "
        f"{question_percent[person]:.2f}%"
    )

PERSONALITY ANALYSIS

Aman
----------------------------------------
Average burst       : 2.72
Caring keywords     : 99
Night messages      : 79.80%
Average words/msg   : 5.02
Uppercase messages  : 0.00%
Silent days         : 0.00%
Funny messages      : 0.00%
Questions           : 6.61%

Karan
----------------------------------------
Average burst       : 1.23
Caring keywords     : 29
Night messages      : 2.26%
Average words/msg   : 57.05
Uppercase messages  : 0.00%
Silent days         : 0.00%
Funny messages      : 0.00%
Questions           : 0.00%

Neha
----------------------------------------
Average burst       : 2.57
Caring keywords     : 23
Night messages      : 4.72%
Average words/msg   : 5.32
Uppercase messages  : 63.30%
Silent days         : 0.00%
Funny messages      : 0.00%
Questions           : 6.25%

Priya
----------------------------------------
Average burst       : 1.66
Caring keywords     : 621
Night messages      : 1.25%
Average words/msg   : 5.00
Uppercase messages  :

In [15]:
# ==========================================
# FINAL PERSONALITY ARCHETYPE
# ==========================================

archetypes = {}

for person in participants:

    scores = {}

    # 1. THE SPAMMER
    if average_burst[person] > 3:
        scores["THE SPAMMER"] = average_burst[person]

    # 2. THE GROUP MOM
    if caring_counts[person] == max(caring_counts.values()):
        scores["THE GROUP MOM"] = caring_counts[person]

    # 3. THE NIGHT OWL
    if night_percent[person] > 60:
        scores["THE NIGHT OWL"] = night_percent[person]

    # 4. THE STORYTELLER
    if average_words[person] > 30:
        scores["THE STORYTELLER"] = average_words[person]

    # 5. THE DRAMA QUEEN
    if uppercase_percent[person] > 30 or exclamation_counts[person] >= 2:
        scores["THE DRAMA QUEEN"] = uppercase_percent[person]

    # 6. THE GHOST
    if silent_percent[person] > 60:
        scores["THE GHOST"] = silent_percent[person]

    # 7. THE COMEDIAN
    if comedian_percent[person] == max(comedian_percent.values()):
        scores["THE COMEDIAN"] = comedian_percent[person]

    # 8. THE QUESTION MASTER
    if question_percent[person] > 25:
        scores["THE QUESTION MASTER"] = question_percent[person]

    # If at least one rule is satisfied
    if len(scores) > 0:
        archetypes[person] = max(scores, key=scores.get)
    else:
        archetypes[person] = "NO CLEAR ARCHETYPE"


# ------------------------------------------
# FINAL ARCHETYPE TABLE
# ------------------------------------------

print("GROUPDNA PERSONALITY ARCHETYPES")
print("=" * 65)
print(f"{'Participant':<15} {'Archetype':<30}")
print("-" * 65)

for person in sorted(archetypes):
    print(
        f"{person:<15} "
        f"{archetypes[person]:<30}"
    )

GROUPDNA PERSONALITY ARCHETYPES
Participant     Archetype                     
-----------------------------------------------------------------
Aman            THE NIGHT OWL                 
Karan           THE STORYTELLER               
Neha            THE DRAMA QUEEN               
Priya           THE GROUP MOM                 
Rahul           THE SPAMMER                   
Vikas           THE GHOST                     


In [16]:
# ==========================================
# FINAL GROUPDNA REPORT
# ==========================================

print()
print("=" * 75)
print("                 GROUPDNA - FINAL REPORT")
print("=" * 75)

print()
print("GROUP OVERVIEW")
print("-" * 75)
print(f"Total messages       : {total_messages}")
print(f"Date range           : {start_date} to {end_date}")
print(f"Total days           : {total_days}")
print(f"Participants         : {len(participants)}")
print(f"System messages      : {system_messages}")
print(f"Media messages       : {media_messages}")
print(f"Deleted messages     : {deleted_messages}")

print()
print("MESSAGE COUNT BY PARTICIPANT")
print("-" * 75)
print(f"{'Participant':<15}{'Messages':>12}{'Share':>15}")

for person, count in sorted_counts:
    share = (count / total_messages) * 100
    print(f"{person:<15}{count:>12}{share:>14.2f}%")

print()
print("MOST ACTIVE PERIOD")
print("-" * 75)
print(f"Busiest day         : {busiest_day}")
print(f"Messages on day     : {busiest_day_count}")
print(
    f"Busiest hour        : "
    f"{busiest_hour:02d}:00 - {busiest_hour + 1:02d}:00"
)
print(f"Messages in hour    : {busiest_hour_count}")

print()
print("LONGEST SILENT STREAK")
print("-" * 75)

for person in sorted(
    silent_streaks,
    key=silent_streaks.get,
    reverse=True
):
    print(f"{person:<15}: {silent_streaks[person]} days")

print()
print("PERSONALITY ARCHETYPES")
print("-" * 75)
print(f"{'Participant':<15}{'Archetype':<35}")

for person in sorted(archetypes):
    print(
        f"{person:<15}"
        f"{archetypes[person]:<35}"
    )

print()
print("=" * 75)
print("                 END OF GROUPDNA REPORT")
print("=" * 75)


                 GROUPDNA - FINAL REPORT

GROUP OVERVIEW
---------------------------------------------------------------------------
Total messages       : 3174
Date range           : 2024-04-01 to 2024-05-30
Total days           : 60
Participants         : 6
System messages      : 4
Media messages       : 32
Deleted messages     : 15

MESSAGE COUNT BY PARTICIPANT
---------------------------------------------------------------------------
Participant        Messages          Share
Rahul                   953         30.03%
Priya                   718         22.62%
Neha                    635         20.01%
Aman                    490         15.44%
Karan                   354         11.15%
Vikas                    24          0.76%

MOST ACTIVE PERIOD
---------------------------------------------------------------------------
Busiest day         : 2024-05-04
Messages on day     : 76
Busiest hour        : 18:00 - 19:00
Messages in hour    : 248

LONGEST SILENT STREAK
----------------